# Testing del Clasificador Multi-Score (Cobranzas - V2)
Este notebook evalúa el rendimiento del algoritmo usando la configuración exacta proporcionada de categorías (`Promesa de Pago`, `Riesgo Legal`, `Vulnerabilidad Económica`, `Localizar Fallido`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from classifier_multiscore import MultiCategoryScoreClassifier

In [ ]:
# 1. Instanciar el clasificador usando el JSON actualizado
clasificador = MultiCategoryScoreClassifier.from_json("cobranzas_config_v2.json")
print("Clasificador Multi-Score inicializado correctamente.")

In [ ]:
# 2. Cargar el dataset de pruebas actualizado
df = pd.read_csv("dataset_cobranzas_v2.csv")
display(df.head())

In [ ]:
# 3. Aplicar el clasificador al DataFrame
def aplicar_clasificador(texto):
    res = clasificador.evaluate_text(texto)
    return pd.Series([
        res['categoria_principal'], 
        res['scores_por_categoria'],
        res['evidencia_detallada']
    ])

df[['categoria_predicha', 'scores_detalle', 'evidencia']] = df['transcripcion'].apply(aplicar_clasificador)
display(df.head())

In [ ]:
# 4. Evaluar el rendimiento (Métricas)
df['acierto'] = df['categoria_real'] == df['categoria_predicha']
accuracy = df['acierto'].mean()
print(f"\nPrecisión Global (Accuracy): {accuracy * 100:.2f}%\n")

print("--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(df['categoria_real'], df['categoria_predicha']))

In [ ]:
# 5. Matriz de Confusión
labels = sorted(df['categoria_real'].unique())
pred_labels = df['categoria_predicha'].unique()
all_labels = sorted(list(set(labels) | set(pred_labels)))

cm = confusion_matrix(df['categoria_real'], df['categoria_predicha'], labels=all_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=all_labels, yticklabels=all_labels)
plt.xlabel('Categoría Predicha (Algoritmo)')
plt.ylabel('Categoría Real (Ground Truth)')
plt.title('Matriz de Confusión - Multi-Score Classifier')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Mostrar los errores para iterar el JSON
errores = df[~df['acierto']]
if not errores.empty:
    print("\n--- FALSOS NEGATIVOS (Frases que no alcanzaron el score necesario) ---")
    display(errores[['transcripcion', 'categoria_real', 'categoria_predicha', 'scores_detalle']].head(15))
else:
    print("\n¡El clasificador acertó el 100% de los casos!")